# ADEPT — Fine-tuning de BETO sobre EmoEvent_es (EmoEvalEs) · v3

**Objetivo:** entrenar el clasificador de emociones de ADEPT con la configuración que define la tesis: BETO ajustado sobre las 5.723 instancias de entrenamiento de EmoEvent_es, **siete categorías nativas** (ira, asco, miedo, alegría, tristeza, sorpresa y otros), hasta 128 tokens, lotes de 16, AdamW, tasa de aprendizaje 2×10⁻⁵, máximo 5 épocas, semilla fija y detención temprana tras dos épocas sin mejora de la F1 macro de desarrollo.

**Cómo se usa, en dos tiempos (así se respeta que el test se use una sola vez):**
1. **Corridas de ajuste** (`FINAL_RUN = False`, el valor por defecto): entrena, evalúa **solo en desarrollo** y registra la corrida en `runs_log.csv`. Si querés probar variantes (por ejemplo `CLASS_WEIGHTS = "sqrt"`), cambiá `RUN_NAME` y el parámetro, y compará las filas de `runs_log.csv`. El test no se toca.
2. **Corrida final** (`FINAL_RUN = True`): con la configuración elegida en desarrollo, entrena una vez más, evalúa el test **una sola vez**, verifica el criterio F1 macro ≥ 0,70 y guarda todo.

**Qué produce la corrida final** (en Drive, `ADEPT/beto-emoevent/`): `model/`, `preprocessing.py`, `metrics.json`, `test_predictions.csv`, `confusion_matrix_test.png`, `training_log_<corrida>.csv`, `runs_log.csv` y un `.zip`.

**Cómo correrlo:** *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)* y *Ejecutar todo*. Estimación: unos 4 minutos por corrida.

**Advertencias:**
1. **El archivo público de test está corrupto y se repara.** Un registro contiene otros 30 fusionados, y la columna de etiquetas quedó corrida 30 posiciones para casi todo el test. El notebook recupera los 31 registros fusionados, realinea las etiquetas, deja el test en **las 1.656 filas oficiales** y lo verifica contra el corpus completo. Si algo no cuadra, se detiene.
2. **Dominio:** el corpus son tuits de abril de 2019, no conversaciones de adultos mayores. Una buena métrica acá no garantiza el mismo rendimiento en ADEPT (la tesis ya lo reconoce como limitación).
3. **No ajustes la configuración mirando el test.** Cualquier variante se elige solo con desarrollo y queda registrada.

**Referencias:** Plaza-del-Arco et al. (2020), *EmoEvent*, LREC 2020. Plaza-del-Arco et al. (2021), *Overview of the EmoEvalEs task*, Procesamiento del Lenguaje Natural 67. Cañete et al. (2020), BETO.

## Paso 1 — Preparar el entorno
Se instalan las librerías con versión fija de `transformers`, para que el notebook funcione igual aunque salgan versiones nuevas. Si Colab te pide reiniciar el entorno, reiniciá y volvé a correr desde esta celda.

In [ ]:
!pip install -q "transformers==4.57.6" "accelerate>=1.0" "emoji>=2.0" scikit-learn matplotlib pandas

import sys, torch, transformers
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("ATENCION: no hay GPU. Menú: Entorno de ejecución -> Cambiar tipo de entorno -> GPU T4. Sin GPU el entrenamiento es muy lento.")

## Paso 2 — Configuración
Todos los parámetros están acá, en un solo lugar. Para probar variantes, cambiar estos valores y volver a correr desde esta celda (guardar los resultados de cada corrida: las variantes que pruebes sirven para la tesis).

In [ ]:
import os, re, json, random, platform, datetime, time
from pathlib import Path

# ---- Modelos ----
BASE_MODEL = "dccuchile/bert-base-spanish-wwm-cased"    # BETO (con mayúsculas)
BASELINE_MODEL = "UMUTeam/beto-base-emotion-analysis"    # antecedente publicado, solo para comparar (opcional)

# ---- Configuración inicial de la tesis (sección 3.3.3) ----
SEED = 42
MAX_LEN = 128             # tokens por texto
EPOCHS = 5                # máximo; la detención temprana puede cortar antes
BATCH_TRAIN = 16
BATCH_EVAL = 64
LEARNING_RATE = 2e-5
EARLY_STOP_PATIENCE = 2   # épocas sin mejora de la F1 macro de desarrollo

# ---- Valores que la tesis no fija (se dejan en el valor por defecto de la librería) ----
WEIGHT_DECAY = 0.0
WARMUP_RATIO = 0.0
CLASS_WEIGHTS = None      # None | "sqrt" | "balanced"  (variante a probar solo con desarrollo)
USE_EMOJI_TEXT = True     # convierte emojis a palabras; decisión de preprocesamiento a describir en la tesis

# ---- Flujo de trabajo ----
RUN_NAME = "base"         # nombre de la corrida en runs_log.csv; cambialo al probar variantes
FINAL_RUN = False         # True solo en la corrida final: ahí se evalúa el test (una sola vez)
ACCEPT_MACRO_F1 = 0.70    # criterio de aceptación de la tesis (RF10, Tabla 3.XII)
RUN_BASELINE = False      # compara contra el modelo publicado (solo cuenta si FINAL_RUN = True)
BASELINE_ID2LABEL = None  # si el modelo publicado usa nombres genéricos (LABEL_0...), indicá {0: "anger", ...}
BASELINE_COLLAPSE = {"neutral": "others"}   # el modelo publicado tiene 8 clases; 'neutral' se cuenta como 'others'

# ---- Datos (repositorio oficial del corpus) ----
GITHUB_RAW = "https://raw.githubusercontent.com/fmplaza/EmoEvent/master/splits/es"
EXPECTED = {"train": 5723, "dev": 844, "test": 1656}
FULL_CORPUS_URL = "https://raw.githubusercontent.com/fmplaza/EmoEvent/master/emoevent_es.csv"   # corpus completo, para verificar

# Orden fijo de las clases (no cambiarlo: el modelo guarda este orden)
LABELS = ["anger", "disgust", "fear", "joy", "others", "sadness", "surprise"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

## Paso 3 — Dónde se guardan los resultados
En Colab monta Google Drive (te va a pedir permiso). Los resultados quedan en `Mi unidad/ADEPT/beto-emoevent/`.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/ADEPT/beto-emoevent")
except ImportError:
    OUT_DIR = Path("./adept_beto_output")   # fuera de Colab (pruebas locales)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Los resultados se guardan en:", OUT_DIR.resolve())

## Paso 4 — Descargar, reparar y verificar los datos
Se descargan las particiones oficiales del repositorio del corpus.

**Defecto del archivo de test:** una fila contiene fusionados otros 30 registros (con los tabuladores convertidos en espacios), y al final del archivo quedan 30 filas con solo la etiqueta. La columna de etiquetas conservó todas sus filas mientras la de textos perdió 30, así que **desde esa fila en adelante cada texto quedó con la etiqueta de 30 posiciones más abajo**. La celda:
1. realinea las etiquetas de esas filas,
2. **recupera los 31 registros fusionados** separándolos por sus identificadores (sus etiquetas están en las posiciones de la propia fila fusionada),
3. comprueba que el conjunto de etiquetas quedó intacto (las mismas 1.656 etiquetas originales),
4. **verifica contra el corpus completo** (`emoevent_es.csv`): para los tuits que se pueden emparejar, la etiqueta tiene que coincidir (≥ 99 %). Si no coincide, el notebook se detiene.

In [ ]:
import csv, urllib.request
import pandas as pd

DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)

def download(url, path):
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path

def read_tsv(path):
    # QUOTE_NONE: los tuits tienen comillas sueltas que confundirían al lector por defecto.
    return pd.read_csv(path, sep="\t", quoting=csv.QUOTE_NONE, encoding="utf-8")

raw = {s: read_tsv(download(f"{GITHUB_RAW}/{s}.tsv", DATA_DIR / f"{s}.tsv")) for s in ["train", "dev", "test"]}

UUID_RE = re.compile(r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}")
UUID_SPLIT = re.compile("(" + UUID_RE.pattern + ")")
COLS = ["id", "event", "tweet", "offensive"]

def split_fused_row(row):
    # La fila fusionada trae los registros pegados con espacios en lugar de tabuladores:
    #   <tweet0> NO <id1> <evento1> <tweet1> NO <id2> ... <idN> <eventoN> <tweetN>
    # y el campo 'offensive' del último registro quedó en su propia columna.
    parts = UUID_SPLIT.split(str(row["tweet"]))      # [texto0, id1, resto1, id2, resto2, ...]
    m0 = re.match(r"^(.*?)\s+(NO|OFF)\s*$", parts[0], re.S)
    recs = [{"id": row["id"], "event": row["event"], "tweet": (m0.group(1) if m0 else parts[0]).strip(),
             "offensive": m0.group(2) if m0 else row["offensive"]}]
    for k in range(1, len(parts), 2):
        uid, rest = parts[k], parts[k + 1]
        if k == len(parts) - 2:                      # último registro: su 'offensive' está en la columna propia
            m = re.match(r"^\s+(\S+)\s+(.*?)\s*$", rest, re.S)
            assert m, "no se pudo separar el último registro fusionado"
            event, tweet, off = m.group(1), m.group(2), row["offensive"]
        else:
            m = re.match(r"^\s+(\S+)\s+(.*?)\s+(NO|OFF)\s*$", rest, re.S)
            assert m, f"no se pudo separar el registro fusionado {uid}"
            event, tweet, off = m.group(1), m.group(2), m.group(3)
        recs.append({"id": uid, "event": event, "tweet": tweet.strip(), "offensive": off})
    return recs

def repair_split(df):
    # Detecta el defecto y devuelve (df_reparado, info). Si no hay defecto, no toca nada.
    n_uuid = df["tweet"].fillna("").map(lambda t: len(UUID_RE.findall(t)))
    n_empty = int(df["tweet"].isna().sum())
    info = {"fused_rows": int((n_uuid > 0).sum()), "empty_text_rows": n_empty, "label_shift": 0, "recovered": 0}
    if info["fused_rows"] == 0 and n_empty == 0:
        return df.copy(), info
    shift = int(n_uuid.max())                       # cuántos registros extra quedaron fusionados
    f = int(n_uuid.values.argmax())                 # posición de la fila fusionada
    assert info["fused_rows"] == 1 and n_empty == shift and df["tweet"].iloc[-shift:].isna().all(), \
        "Patrón de corrupción inesperado: revisar el archivo a mano antes de seguir."
    labels = df["emotion"].tolist()
    rows = [{**df.iloc[i][COLS].to_dict(), "emotion": labels[i]} for i in range(f)]            # antes de la fusión: sin cambios
    fused = split_fused_row(df.iloc[f])
    assert len(fused) == shift + 1, f"se esperaban {shift + 1} registros fusionados y se separaron {len(fused)}"
    rows += [{**rec, "emotion": labels[f + k]} for k, rec in enumerate(fused)]                 # las etiquetas están en la propia fila
    rows += [{**df.iloc[i][COLS].to_dict(), "emotion": labels[i + shift]}                      # el resto: etiqueta 'shift' filas más abajo
             for i in range(f + 1, len(df) - shift)]
    info.update(label_shift=shift, fused_row_index=f, recovered=len(fused))
    return pd.DataFrame(rows), info

data, repair = {}, {}
for s, df in raw.items():
    data[s], repair[s] = repair_split(df)
    assert data[s]["id"].is_unique, f"ids repetidos en {s}"
    assert set(data[s]["emotion"]) <= set(LABELS), f"etiquetas desconocidas en {s}"
    assert sorted(data[s]["emotion"]) == sorted(raw[s]["emotion"]), f"el conjunto de etiquetas de {s} cambió"
    assert len(data[s]) == EXPECTED[s], f"{s}: {len(data[s])} filas, se esperaban {EXPECTED[s]}"

print(f"{'split':6s}{'leídas':>8s}{'usadas':>8s}{'esperadas':>11s}{'corrimiento':>13s}{'recuperadas':>13s}")
for s in data:
    print(f"{s:6s}{len(raw[s]):>8d}{len(data[s]):>8d}{EXPECTED[s]:>11d}{repair[s]['label_shift']:>13d}{repair[s]['recovered']:>13d}")
assert repair["train"]["recovered"] == 0 and repair["dev"]["recovered"] == 0, "train/dev no deberían tener defectos"
if repair["test"]["recovered"]:
    print(f"\nAVISO: se reparó el test (corrimiento de {repair['test']['label_shift']} filas) y se recuperaron "
          f"{repair['test']['recovered']} registros fusionados. Documentarlo en la tesis.")

# Sin filtración entre particiones (mismo id o mismo texto en dos particiones)
for a, b in [("train", "dev"), ("train", "test"), ("dev", "test")]:
    assert not (set(data[a]["id"]) & set(data[b]["id"])), f"ids compartidos {a}/{b}"
    assert not (set(data[a]["tweet"]) & set(data[b]["tweet"])), f"textos compartidos {a}/{b}"

# ---- Verificación cruzada contra el corpus completo ----
def norm_key(t):
    t = re.sub(r"https?://\S+", "URL", str(t))
    t = re.sub(r"#\w+", "HASHTAG", t)
    t = re.sub(r"@\w+", "USER", t)
    return re.sub(r"\s+", " ", t).strip().lower()

full = read_tsv(download(FULL_CORPUS_URL, DATA_DIR / "emoevent_es.tsv"))
full_label = full.assign(k=full["tweet"].map(norm_key)).drop_duplicates("k", keep=False).set_index("k")["emotion"]

def agreement(df):
    k = df["tweet"].map(norm_key)
    ok = k.isin(full_label.index)
    return int(ok.sum()), float((df.loc[ok, "emotion"].values == full_label.reindex(k[ok]).values).mean())

integrity = {}
print("\nVerificación contra el corpus completo (etiqueta de cada partición vs. etiqueta del corpus):")
for s, df in data.items():
    n, agree = agreement(df)
    integrity[s] = {"matched": n, "label_agreement": round(agree, 4)}
    print(f"  {s:6s} emparejados {n:5d}/{len(df)}  coincidencia {agree:.1%}")
    assert n > 100 and agree >= 0.99, (f"Las etiquetas de '{s}' no coinciden con el corpus completo ({agree:.1%}). "
                                       "No entrenar ni evaluar hasta entender por qué.")
if repair["test"]["recovered"]:
    rec_ids = set(split_fused_row(raw["test"].iloc[repair["test"]["fused_row_index"]])[k]["id"] for k in range(repair["test"]["recovered"]))
    n, agree = agreement(data["test"][data["test"]["id"].isin(rec_ids)])
    integrity["test_recovered_rows"] = {"matched": n, "label_agreement": round(agree, 4)}
    print(f"  (los {len(rec_ids)} registros recuperados: emparejados {n}, coincidencia {agree:.1%})")
    assert n == 0 or agree >= 0.99, "Las etiquetas de los registros recuperados no coinciden con el corpus completo."
print("Datos verificados.")

dist = pd.DataFrame({s: data[s]["emotion"].value_counts().reindex(LABELS) for s in data})
dist_pct = (dist / dist.sum() * 100).round(1)
print("\nDistribución de clases (cantidad):"); print(dist)
print("\nDistribución de clases (%):"); print(dist_pct)

**Qué mirar:** las clases están muy desbalanceadas (`others` y `joy` dominan; `fear` y `disgust` son muy pocas). Por eso la métrica de aceptación es la **F1 macro** (promedia todas las clases por igual) y se reporta además el accuracy. Por defecto el entrenamiento no usa pesos por clase, como en la configuración de la tesis; si querés probarlos, es una variante (`CLASS_WEIGHTS`) que se evalúa solo con desarrollo.

## Paso 5 — Preprocesamiento
Dos cambios mínimos al texto: las URLs se reemplazan por la palabra `http`, y los emojis se convierten a palabras en español (BETO no conoce los emojis y los vería como "desconocido", perdiendo información emocional). **Esta función se guarda en `preprocessing.py`: el servicio de producción tiene que usar exactamente el mismo código**, si no el modelo recibiría textos distintos de los que vio al entrenar.

In [ ]:
PREPROCESS_SRC = r'''import re
import emoji

_URL_RE = re.compile(r"https?://\S+")

def preprocess(text, demojize=True):
    # Normaliza un texto antes de tokenizarlo (mismo código en entrenamiento y producción).
    text = str(text)
    text = _URL_RE.sub("http", text)
    if demojize:
        text = emoji.demojize(text, language="es", delimiters=(" ", " "))
        text = text.replace("_", " ")
    return re.sub(r"\s+", " ", text).strip()
'''
exec(PREPROCESS_SRC)                                              # define preprocess() en este notebook
(OUT_DIR / "preprocessing.py").write_text(PREPROCESS_SRC, encoding="utf-8")   # y la guarda para el backend

for s in data:
    data[s]["text"] = data[s]["tweet"].map(lambda t: preprocess(t, USE_EMOJI_TEXT))

for i in range(3):
    print("ANTES  :", data["train"]["tweet"].iloc[i][:140])
    print("DESPUES:", data["train"]["text"].iloc[i][:140], "\n")

## Paso 6 — Tokenización
BETO no lee texto directamente: lo parte en "tokens". Se verifica que `MAX_LEN` alcance para casi todos los textos.

In [ ]:
import numpy as np
import torch
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

lens = np.array([len(x) for x in tokenizer(data["train"]["text"].tolist())["input_ids"]])
print("Largo en tokens -> mediana:", int(np.median(lens)), "| p95:", int(np.percentile(lens, 95)),
      "| máximo:", int(lens.max()), "| truncados con MAX_LEN=%d: %.2f%%" % (MAX_LEN, (lens > MAX_LEN).mean() * 100))

class EmoDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.enc, self.labels = encodings, labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

def make_ds(df):
    enc = tokenizer(df["text"].tolist(), truncation=True, max_length=MAX_LEN)
    return EmoDataset(enc, df["emotion"].map(LABEL2ID).tolist())

ds = {s: make_ds(data[s]) for s in data}
y = {s: data[s]["emotion"].map(LABEL2ID).to_numpy() for s in data}
collator = DataCollatorWithPadding(tokenizer)
print({s: len(ds[s]) for s in ds})

## Paso 7 — Métricas
- **accuracy:** porcentaje de aciertos (es la métrica del ranking oficial de EmoEvalEs).
- **F1 macro:** promedio del F1 de cada emoción, sin importar cuántos ejemplos tenga. Es la métrica de aceptación de la tesis (RF10: ≥ 0,70 en la partición oficial de prueba).
- **F1 ponderado:** como el macro, pero ponderado por cantidad de ejemplos.

Como referencia, el mejor sistema del ranking oficial de EmoEvalEs 2021 logró accuracy 0,7277 (portal ODESIA). Esa tabla trae también una columna "MacroF1" de 0,7170 que no se puede comparar con la F1 macro de acá sin confirmar cómo la calcularon (está muy cerca del F1 ponderado). Se midió sobre el test completo.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib
import matplotlib.pyplot as plt

def metrics_from_ids(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return metrics_from_ids(labels, np.argmax(logits, axis=-1))

def per_class_report(y_true, y_pred, names):
    return classification_report(y_true, y_pred, labels=list(range(len(names))), target_names=names,
                                 digits=3, zero_division=0, output_dict=True)

def plot_confusion(y_true, y_pred, names, title, path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(names))), normalize="true")
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay(cm, display_labels=names).plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
    ax.set_title(title); plt.xticks(rotation=45, ha="right"); plt.tight_layout()
    fig.savefig(path, dpi=150); plt.show()

## Paso 8 — Entrenamiento
Se entrena con *early stopping*: después de cada época se mide el macro-F1 en el conjunto **dev** y se conserva la mejor época. El conjunto **test** no se toca hasta el final.

Es normal ver una advertencia de que algunos pesos "no fueron inicializados": es la capa de clasificación nueva, que justamente se entrena ahora.

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback, set_seed

set_seed(SEED)

# Pesos por clase: las clases chicas pesan más en la pérdida para que el modelo no las ignore.
counts = np.bincount(y["train"], minlength=len(LABELS))
balanced = len(y["train"]) / (len(LABELS) * counts)
if CLASS_WEIGHTS == "balanced":
    w = balanced
elif CLASS_WEIGHTS == "sqrt":
    w = np.sqrt(balanced)
else:
    w = None
class_weights = torch.tensor(w, dtype=torch.float) if w is not None else None
print("Pesos por clase:", {l: round(float(x), 2) for l, x in zip(LABELS, w)} if w is not None else "sin pesos")

class WeightedTrainer(Trainer):
    # Igual que Trainer, pero la pérdida usa los pesos por clase.
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        wt = self.class_weights.to(outputs.logits.device) if self.class_weights is not None else None
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=wt)
        return (loss, outputs) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID)

args = TrainingArguments(
    output_dir="checkpoints",                 # local de Colab (los checkpoints pesan mucho; el modelo final va a Drive)
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    optim="adamw_torch",                      # AdamW (sección 3.3.3 de la tesis)
    data_seed=SEED,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    logging_strategy="epoch",
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=ds["train"], eval_dataset=ds["dev"],
    processing_class=tokenizer, data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)],
    class_weights=class_weights,
)

t0 = time.time()
trainer.train()
train_minutes = (time.time() - t0) / 60
print(f"\nEntrenamiento terminado en {train_minutes:.1f} minutos.")

log = pd.DataFrame(trainer.state.log_history)
log.to_csv(OUT_DIR / f"training_log_{RUN_NAME}.csv", index=False)
cols = [c for c in ["epoch", "loss", "eval_loss", "eval_accuracy", "eval_macro_f1"] if c in log.columns]
print(log[cols].groupby("epoch").first().round(4))

**Cómo leer la tabla:** `loss` es el error en entrenamiento y `eval_loss` el error en dev. Si `loss` sigue bajando pero `eval_loss` sube, el modelo empieza a memorizar (sobreajuste); el early stopping está pensado para frenar ahí.

## Paso 9 — Resultados
**Siempre** se evalúa el conjunto de desarrollo y se registra la corrida en `runs_log.csv`. El **test solo se evalúa si `FINAL_RUN = True`**, y una sola vez. En la corrida final se verifica el criterio de aceptación y se guardan las predicciones y la matriz de confusión.

In [ ]:
def predict_split(name):
    out = trainer.predict(ds[name])
    logits = out.predictions
    return logits.argmax(axis=-1), torch.softmax(torch.tensor(logits), dim=-1).numpy()

# ---- Desarrollo (siempre) ----
dev_pred, dev_prob = predict_split("dev")
dev_metrics = metrics_from_ids(y["dev"], dev_pred)
print(f"DEV   accuracy={dev_metrics['accuracy']:.4f}  macro-F1={dev_metrics['macro_f1']:.4f}  weighted-F1={dev_metrics['weighted_f1']:.4f}")
print("\nDetalle por emoción (DEV):")
print(classification_report(y["dev"], dev_pred, labels=list(range(len(LABELS))), target_names=LABELS, digits=3, zero_division=0))

# ---- Registro de la corrida (siempre) ----
runs_path = OUT_DIR / "runs_log.csv"
previous = pd.read_csv(runs_path) if runs_path.exists() else pd.DataFrame()
if FINAL_RUN and len(previous) and "final" in previous.columns and previous["final"].astype(bool).any():
    print("\nATENCION: ya hay una corrida final registrada. Volver a evaluar el test invalida el criterio de 'una sola vez' "
          "(tesis, sección 3.3.3). Si lo hacés igual, documentalo.")
ev = log.dropna(subset=["eval_macro_f1"]) if "eval_macro_f1" in log.columns else pd.DataFrame()
best_epoch = float(ev.loc[ev["eval_macro_f1"].idxmax(), "epoch"]) if len(ev) else None
run_row = {"run_name": RUN_NAME, "timestamp": datetime.datetime.now().isoformat(timespec="seconds"), "final": bool(FINAL_RUN),
           "learning_rate": LEARNING_RATE, "batch_train": BATCH_TRAIN, "epochs_max": EPOCHS, "seed": SEED,
           "class_weights": str(CLASS_WEIGHTS), "weight_decay": WEIGHT_DECAY, "warmup_ratio": WARMUP_RATIO,
           "demojize": USE_EMOJI_TEXT, "best_epoch": best_epoch, "train_minutes": round(train_minutes, 1),
           "dev_accuracy": round(dev_metrics["accuracy"], 4), "dev_macro_f1": round(dev_metrics["macro_f1"], 4),
           "dev_weighted_f1": round(dev_metrics["weighted_f1"], 4)}
pd.concat([previous, pd.DataFrame([run_row])], ignore_index=True).to_csv(runs_path, index=False)
print("\nCorrida registrada en", runs_path)
print(pd.read_csv(runs_path)[["run_name", "final", "learning_rate", "batch_train", "class_weights", "best_epoch",
                              "dev_accuracy", "dev_macro_f1"]].to_string(index=False))

results = {"dev": {"pred": dev_pred, "prob": dev_prob, "metrics": dev_metrics}}

if not FINAL_RUN:
    print("\nCorrida de AJUSTE: el conjunto de test NO se evaluó. Compará variantes solo con DEV (tabla de arriba). "
          "Cuando elijas la configuración definitiva, poné FINAL_RUN = True y corré todo de nuevo.")
else:
    # ---- Test (una sola vez) ----
    test_pred, test_prob = predict_split("test")
    results["test"] = {"pred": test_pred, "prob": test_prob, "metrics": metrics_from_ids(y["test"], test_pred)}
    m = results["test"]["metrics"]
    print(f"\nTEST  accuracy={m['accuracy']:.4f}  macro-F1={m['macro_f1']:.4f}  weighted-F1={m['weighted_f1']:.4f}")
    gap = dev_metrics["accuracy"] - m["accuracy"]
    if gap > 0.15:
        print(f"ALERTA: el accuracy en test es {gap:.0%} menor que en dev. Con particiones de la misma distribución eso casi seguro es un problema de datos. Revisar el paso 4.")
    print("\nDetalle por emoción (TEST):")
    print(classification_report(y["test"], test_pred, labels=list(range(len(LABELS))), target_names=LABELS, digits=3, zero_division=0))
    plot_confusion(y["test"], test_pred, LABELS, "Matriz de confusión (test, normalizada por fila)", OUT_DIR / "confusion_matrix_test.png")
    pred_df = data["test"][["id", "event", "tweet", "text", "emotion"]].copy()
    pred_df["predicted"] = [ID2LABEL[i] for i in test_pred]
    pred_df["confidence"] = test_prob.max(axis=1).round(4)
    pred_df.to_csv(OUT_DIR / "test_predictions.csv", index=False)
    meets = m["macro_f1"] >= ACCEPT_MACRO_F1
    print(f"\nCRITERIO DE ACEPTACIÓN (RF10): F1 macro >= {ACCEPT_MACRO_F1:.2f} -> {'CUMPLE' if meets else 'NO CUMPLE'} (F1 macro = {m['macro_f1']:.4f})")
    if not meets:
        print("El resultado se informa tal cual. No se ajusta la configuración mirando el test: cualquier cambio se decide con desarrollo y se documenta.")

**Cómo leer la matriz (corrida final):** cada fila es la emoción real y cada columna lo que predijo el modelo; la diagonal es lo acertado. Es esperable que `fear`, `disgust` y `surprise` salgan flojas: tienen pocos ejemplos.

## Paso 10 — Comparación con el modelo publicado (opcional, apagado por defecto)
Si `RUN_BASELINE = True` y `FINAL_RUN = True`, se evalúa sobre el mismo test el modelo `UMUTeam/beto-base-emotion-analysis`. Ese modelo tiene **8 clases** (separa `neutral` de `others`), así que no es una comparación directa: `neutral` se cuenta como `others` (configurable en `BASELINE_COLLAPSE`).

In [ ]:
def predict_texts(mdl, tok, texts, batch_size=64):
    # Devuelve probabilidades (n_textos x n_clases) para una lista de textos.
    mdl.eval(); device = next(mdl.parameters()).device; out = []
    for i in range(0, len(texts), batch_size):
        batch = tok(texts[i:i + batch_size], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            out.append(torch.softmax(mdl(**batch).logits, dim=-1).cpu().numpy())
    return np.concatenate(out)

baseline = None
if RUN_BASELINE and FINAL_RUN:
    try:
        b_tok = AutoTokenizer.from_pretrained(BASELINE_MODEL)
        b_mdl = AutoModelForSequenceClassification.from_pretrained(BASELINE_MODEL).to("cuda" if torch.cuda.is_available() else "cpu")
        src = BASELINE_ID2LABEL if BASELINE_ID2LABEL else b_mdl.config.id2label
        b_names = [str(src[i]).lower() for i in range(b_mdl.config.num_labels)]
        b_names_c = [BASELINE_COLLAPSE.get(n, n) for n in b_names]
        print("Etiquetas del modelo publicado:", b_names, "-> tras colapsar:", b_names_c)
        if not set(b_names_c) <= set(LABELS):
            print("Las etiquetas del modelo publicado no coinciden con las del corpus. Completá BASELINE_ID2LABEL o BASELINE_COLLAPSE en el paso 2.")
        else:
            texts_b = data["test"]["tweet"].map(lambda t: preprocess(t, demojize=False)).tolist()
            b_pred = np.array([LABEL2ID[b_names_c[i]] for i in predict_texts(b_mdl, b_tok, texts_b).argmax(axis=1)])
            baseline = metrics_from_ids(y["test"], b_pred)
            print(f"\n{'modelo':38s}{'accuracy':>10s}{'macro-F1':>10s}{'weighted-F1':>13s}")
            for nm, mm in [("Propio (BETO + EmoEvent)", results["test"]["metrics"]), ("Publicado (UMUTeam)", baseline)]:
                print(f"{nm:38s}{mm['accuracy']:>10.4f}{mm['macro_f1']:>10.4f}{mm['weighted_f1']:>13.4f}")
        del b_mdl
    except Exception as e:
        print("No se pudo evaluar el modelo publicado:", repr(e))
else:
    print("Comparación con el modelo publicado omitida (RUN_BASELINE y FINAL_RUN deben ser True).")

## Paso 11 — Prueba cualitativa con frases conversacionales
Frases parecidas a lo que diría un adulto mayor en ADEPT, más un ejemplo de la **agregación por sesión** de la tesis (RF10): la categoría predominante es la de mayor probabilidad media entre las intervenciones. **No es una métrica**: el modelo se entrenó con tuits, y esta prueba sirve para ver cómo se comporta fuera de ese dominio. 

In [ ]:
frases = [
    "Hoy me levanté con ganas de salir a caminar",
    "Ayer vino mi nieto y pasamos una tarde hermosa",
    "Me siento muy solo desde que se fue mi esposa",
    "No tengo ganas de hablar con nadie",
    "Estoy muy nervioso por el resultado de los estudios",
    "Me da mucha bronca que no me atiendan en la obra social",
    "El otro día me olvidé dónde dejé las llaves y me asusté",
    "Hoy fue un día como cualquier otro",
]
probs = predict_texts(trainer.model, tokenizer, [preprocess(f, USE_EMOJI_TEXT) for f in frases])
for f, p in zip(frases, probs):
    top = p.argsort()[::-1][:2]
    print(f"{f}\n   -> " + " | ".join(f"{ID2LABEL[int(i)]} {p[i]:.2f}" for i in top) + "\n")
mean_p = probs.mean(axis=0)
print(f"Agregación por sesión (frases como una sola sesión): categoría predominante = {ID2LABEL[int(mean_p.argmax())]} "
      f"(probabilidad media {mean_p.max():.2f})")

## Paso 12 — Guardar todo (solo en la corrida final)
Se guarda el modelo entrenado, el tokenizador y un `metrics.json` con la configuración, las versiones (incluida la de `emoji`) y los resultados: lo necesario para que el experimento sea reproducible y citable.

In [ ]:
import shutil, emoji

if not FINAL_RUN:
    print("Corrida de ajuste: no se guarda el modelo ni metrics.json. Para guardarlos, FINAL_RUN = True y correr todo.")
else:
    trainer.save_model(str(OUT_DIR / "model"))          # modelo (safetensors) + configuración con las etiquetas
    tokenizer.save_pretrained(str(OUT_DIR / "model"))
    m = results["test"]["metrics"]
    report = {
        "created_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "run_name": RUN_NAME,
        "base_model": BASE_MODEL,
        "dataset": {"name": "EmoEvent_es (particiones EmoEvalEs)", "source": GITHUB_RAW,
                    "rows_used": {s: len(data[s]) for s in data},
                    "test_repair": repair["test"],
                    "integrity_vs_full_corpus": integrity,
                    "class_counts": {s: dist[s].to_dict() for s in dist}},
        "config": {"seed": SEED, "max_len": MAX_LEN, "epochs_max": EPOCHS, "batch_train": BATCH_TRAIN,
                   "optimizer": "adamw_torch", "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
                   "warmup_ratio": WARMUP_RATIO, "early_stop_patience": EARLY_STOP_PATIENCE,
                   "class_weights": CLASS_WEIGHTS, "demojize": USE_EMOJI_TEXT, "labels_order": LABELS},
        "environment": {"python": platform.python_version(), "torch": torch.__version__,
                        "transformers": transformers.__version__, "emoji": emoji.__version__,
                        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
        "training_minutes": round(train_minutes, 1),
        "best_epoch_dev_macro_f1": float(trainer.state.best_metric) if trainer.state.best_metric is not None else None,
        "results": {
            "dev_7_classes": results["dev"]["metrics"],
            "test_7_classes": m,
            "test_7_classes_per_class": per_class_report(y["test"], results["test"]["pred"], LABELS),
            "acceptance_criterion": {"metric": "macro_f1", "threshold": ACCEPT_MACRO_F1, "value": m["macro_f1"],
                                     "meets": bool(m["macro_f1"] >= ACCEPT_MACRO_F1)},
            "baseline_umuteam_test_7_classes": baseline,
        },
    }
    (OUT_DIR / "metrics.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
    zip_path = shutil.make_archive(str(OUT_DIR.parent / "beto-adept"), "zip", OUT_DIR)
    print("Guardado en:", OUT_DIR.resolve())
    for p in sorted(OUT_DIR.rglob("*")):
        if p.is_file():
            print(f"  {p.relative_to(OUT_DIR)}  ({p.stat().st_size / 1e6:.1f} MB)")
    print("\nZip para descargar:", zip_path)

## Qué sigue
1. **Variantes (solo con desarrollo):** cambiar `RUN_NAME` y un parámetro (`CLASS_WEIGHTS`, `LEARNING_RATE`, `USE_EMOJI_TEXT`), correr con `FINAL_RUN = False` y comparar `runs_log.csv`. Elegir la configuración con la F1 macro de desarrollo.
2. **Corrida final:** con la configuración elegida, `FINAL_RUN = True` y *Ejecutar todo*. Se evalúa el test una vez.
3. **Para la tesis:** informar la F1 macro de test frente al criterio de 0,70 tal como dé (incluida la F1 por categoría y la matriz de confusión), el tratamiento de las 31 filas fusionadas del archivo de test, las variantes probadas con desarrollo y el límite de validez (tuits de 2019 frente a conversaciones de adultos mayores).
4. **Para el backend:** la carpeta `model/` y `preprocessing.py` son lo que consume `ml-service`; las versiones están en `metrics.json`.